In [ ]:
import kagglehub
import os
import splitfolders
from ultralytics import YOLO
from PIL import Image
import cv2

In [ ]:
print("Baixando dataset nos servidores do Kaggle...")
path = kagglehub.dataset_download("muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten")
pasta_origem = os.path.join(path, "Fruit And Vegetable Diseases Dataset")


pasta_destino = "/kaggle/working/dataset_yolo"
pasta_projetos = "/kaggle/working/Modelos_Treinados"


if not os.path.exists(os.path.join(pasta_destino, "train")):
    print("\nOrganizando pastas de treino e validação (Aguarde alguns segundos)...")
    splitfolders.ratio(pasta_origem, output=pasta_destino, seed=1337, ratio=(.8, .2))
else:
    print("\nDataset já organizado.")


model = YOLO("yolo26x-cls.pt")


print("\nIniciando treinamento no Kaggle com T4 x2...")
resultados = model.train(
    data=pasta_destino,
    epochs=100,
    imgsz=224,
    batch=64,
    project=pasta_projetos,
    name='modelo_yolo26x',
    device=[0, 1]
)

In [20]:
imagem_teste = r'H:\Meu Drive\EstudosPython\hackaton_2026\dataset/batata-h2.jpg'

if os.path.exists(imagem_teste):
    print("\nCarregando modelo...")
    modelo_treinado = YOLO(r'H:\Meu Drive\EstudosPython\hackaton_2026/best.pt')
    
    print("Realizando teste rápido na imagem JFIF...")
    try:

        img_aberta = Image.open(imagem_teste).convert('RGB')
        

        predicao = modelo_treinado(img_aberta)

        for r in predicao:
            nome_classe = r.names[r.probs.top1]
            confianca = r.probs.top1conf.item() * 100
            print(f"\n--- RESULTADO ---")
            print(f"Diagnóstico: {nome_classe} | Confiança: {confianca:.2f}%")
            
    except Exception as e:
        print(f"\nErro ao processar a imagem: {e}")
        print("Tente abrir essa imagem no Paint e salvar como .jpg normal.")
else:
    print(f"Erro: O arquivo não foi encontrado neste caminho: {imagem_teste}")


Carregando modelo...
Realizando teste rápido na imagem JFIF...

0: 224x224 Potato__Healthy 1.00, Potato__Rotten 0.00, Jujube__Healthy 0.00, Mango__Healthy 0.00, Grape__Healthy 0.00, 83.2ms
Speed: 11.4ms preprocess, 83.2ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)

--- RESULTADO ---
Diagnóstico: Potato__Healthy | Confiança: 100.00%


In [22]:
caminho_modelo = r'H:\Meu Drive\EstudosPython\hackaton_2026/best.pt'
modelo_treinado = YOLO(caminho_modelo)

cap = cv2.VideoCapture(0)

print("Iniciando câmera... Pressione a tecla 'q' para sair.")

while True:
    ret, frame = cap.read()
    
    if not ret:
        print("Erro ao acessar a câmera.")
        break

    # (O verbose=False esconde os textos técnicos no terminal para não travar a tela)
    predicoes = modelo_treinado(frame, verbose=False)

    # Extrai o resultado do frame atual
    for r in predicoes:
        nome_classe = r.names[r.probs.top1]
        confianca = r.probs.top1conf.item() * 100

        cor = (0, 255, 0) if "Healthy" in nome_classe else (0, 0, 255) 

        texto = f"{nome_classe} | {confianca:.1f}%"
        cv2.putText(
            img=frame, 
            text=texto, 
            org=(10, 50), # Posição x, y na tela
            fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
            fontScale=1, 
            color=cor, 
            thickness=2
        )

    cv2.imshow('Diagnostico Ao Vivo - Hackathon 2026', frame)

    # Trava de segurança para fechar a janela (Aperte a tecla 'q')
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Iniciando câmera... Pressione a tecla 'q' para sair.
